# xbrl-frontier Phase 2: student QLoRA training on a Kaggle T4 (owner-run, resumable)

One run per session. All settings come from `docs/config/thresholds.yaml` in the attached repo snapshot. Set only the three run variables in the next code cell.

**Settings:** Accelerator = GPU T4 (one GPU is used), Internet = on.
**Attach datasets:** `xbrl-frontier-src` (the repo `git archive` tarball) and `xbrl-frontier-corpus` (`train.jsonl` and `dev.jsonl` from `data/corpus/`, checked against `data/corpus_report.json`). When resuming, also attach the previous session's output as a dataset and set `RESUME` to its path.
**After the session:** save the notebook output as a new Kaggle dataset version (it holds `checkpoints/`, `curve/`, `run.json` and, when finished, `final_adapter/`).

In [ ]:
MODEL = 'Qwen/Qwen3-1.7B'
SEED = 1
CURVE = True
VALIDATION = False
RESUME = '/kaggle/input/**/run-Qwen3-1.7B-s1'
SESSION_HOURS = 12
EXPECTED_STEP = 2903
FINNI_TEMPLATE_B64 = None
FINNI_TEMPLATE_SHA256 = None


In [ ]:
import os, time
os.environ.setdefault('XF_SESSION_START', str(time.time()))  # session clock for the T-038 stop (O-034)
import glob, os, shutil, subprocess, sys
dst = '/kaggle/working/xbrl-frontier'
tarballs = glob.glob('/kaggle/input/**/xbrl-frontier-src.tar.gz', recursive=True)
trees = [os.path.dirname(os.path.dirname(os.path.dirname(p)))
         for p in glob.glob('/kaggle/input/**/docs/config/thresholds.yaml', recursive=True)]
if tarballs:
    assert len(tarballs) == 1, tarballs
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['tar', '-xzf', tarballs[0], '-C', dst], check=True)
else:  # Kaggle auto-extracted the upload; attached kernel outputs carry their own repo copy, so prefer the source dataset
    trees = [t for t in trees if 'xbrl-frontier-src' in t] or trees
    assert len(trees) == 1, trees
    shutil.copytree(trees[0], dst, dirs_exist_ok=True)
os.chdir(dst)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import yaml
pins = yaml.safe_load(open('docs/config/thresholds.yaml'))['pins']['software']
missing = [k for k in ('transformers', 'peft', 'bitsandbytes', 'accelerate') if not pins.get(k)]
assert not missing, f'pins.software empty for {missing}: owner approval needed first'
print(pins)

In [ ]:
!pip install -q "transformers=={pins['transformers']}" "peft=={pins['peft']}" "bitsandbytes=={pins['bitsandbytes']}" "accelerate=={pins['accelerate']}" "pyyaml==6.0.3" "pyarrow==25.0.1" "huggingface_hub"
import importlib.metadata as md
installed = {k: md.version(k) for k in ('transformers', 'peft', 'bitsandbytes', 'accelerate')}
bad = {k: (v, pins[k]) for k, v in installed.items() if v != pins[k]}
assert not bad, f'installed != pins.software: {bad}; restart the kernel and re-run this cell'
import torch, transformers, peft, bitsandbytes, accelerate
imported = {'transformers': transformers.__version__, 'peft': peft.__version__,
            'bitsandbytes': bitsandbytes.__version__, 'accelerate': accelerate.__version__}
bad = {k: (v, pins[k]) for k, v in imported.items() if v != pins[k]}
assert not bad, f'imported != pins.software: {bad}; restart the kernel (Run > Restart) and run all again'
print(imported, 'torch', torch.__version__, 'CUDA', torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
def verify_resume_copy(source, output, expected_step):
    import json, shutil
    from pathlib import Path
    source, output = Path(source), Path(output)
    if expected_step is None:
        raise ValueError('EXPECTED_STEP is required for resume')
    assert not output.exists(), f'refusing to reuse output directory: {output}'
    output.mkdir(parents=True, exist_ok=True)
    for sub in ('checkpoints', 'curve'):
        if (source / sub).exists():
            shutil.copytree(source / sub, output / sub, dirs_exist_ok=True)
    shutil.copy2(source / 'run.json', output / 'run.json')
    checkpoints = sorted((output / 'checkpoints').glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1]))
    assert checkpoints and checkpoints[-1].name == f'checkpoint-{expected_step}'
    checkpoint = checkpoints[-1]
    for name in ('adapter_config.json', 'adapter_model.safetensors', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'scaler.pt', 'training_args.bin', 'trainer_state.json'):
        assert (checkpoint / name).is_file() and (checkpoint / name).stat().st_size > 0, name
    state = json.loads((checkpoint / 'trainer_state.json').read_text())
    run = json.loads((output / 'run.json').read_text())
    assert state['global_step'] == run['global_step'] == expected_step < state['max_steps']


In [ ]:
def offline_trainer_command():
    import sys
    code = '''import base64, hashlib, os, runpy
raw = base64.b64decode(os.environ['FINNI_TEMPLATE_B64'], validate=True)
assert hashlib.sha256(raw).hexdigest() == os.environ['FINNI_TEMPLATE_SHA256']
template = raw.decode('utf-8')
assert template.count(chr(0) + 'CONTEXT' + chr(0)) == 1
from xbrl_frontier import sft_format
sft_format.query_template = lambda: template
runpy.run_module('training.kaggle_train', run_name='__main__', alter_sys=True)
'''
    return [sys.executable, '-c', code]

def run_training(command, output, expected_step):
    import json, os, subprocess
    from pathlib import Path
    env = os.environ.copy()
    env.pop('HF_TOKEN', None)
    env['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
    env['FINNI_TEMPLATE_B64'] = FINNI_TEMPLATE_B64
    env['FINNI_TEMPLATE_SHA256'] = FINNI_TEMPLATE_SHA256
    subprocess.run(command, env=env, check=True)
    if expected_step is not None:
        result = json.loads((Path(output) / 'run.json').read_text())
        assert result['sessions'][-1]['resumed_from'] == f'checkpoint-{expected_step}'
        assert result['global_step'] > expected_step, 'resume made no training progress'


In [ ]:
import glob, os
corpus = os.path.dirname(glob.glob('/kaggle/input/**/train.jsonl', recursive=True)[0])
tag = MODEL.split('/')[-1] + f'-s{SEED}' + ('-validation' if VALIDATION else '')
out = f'/kaggle/working/run-{tag}'
cmd = offline_trainer_command() + ['--model', MODEL, '--seed', str(SEED), '--corpus', corpus, '--out', out]
cmd += ['--curve'] if CURVE else []
cmd += ['--validation'] if VALIDATION else []
if RESUME and '*' in RESUME:  # kernel-output mounts vary; resolve to the one directory holding run.json
    hits = sorted({os.path.dirname(f) for f in glob.glob(RESUME.rstrip('/') + '/run.json', recursive=True)})
    assert len(hits) == 1, f'RESUME {RESUME} matched {hits}'
    RESUME = hits[0]
    print('resuming from', RESUME)
if RESUME:
    import hashlib, json
    from pathlib import Path
    from xbrl_frontier import config
    verify_resume_copy(RESUME, out, EXPECTED_STEP)
    previous = json.loads((Path(out) / 'run.json').read_text())
    assert previous['model'] == MODEL and previous['seed'] == SEED and previous['config_hash'] == config.config_hash()
    assert previous['validation_run'] is None and CURVE and not VALIDATION and not previous['complete']
    for split in ('train', 'dev'):
        assert hashlib.sha256((Path(corpus) / f'{split}.jsonl').read_bytes()).hexdigest() == previous['corpus_sha256'][split]
cmd += ['--resume', RESUME] if RESUME else []
cmd += ['--session-hours', str(SESSION_HOURS)] if SESSION_HOURS else []
print(cmd)
run_training(cmd, out, EXPECTED_STEP if RESUME else None)

In [ ]:
!ls -la {out} && cat {out}/run.json | head -60